# Power Outage ML Forecasting — Project Setup & Overview

**Project:** Can Machine Learning Forecast Power Outage Factors?  
**Approach:** Reconstructed CSU-MLP Random Forest framework (Hill et al. 2023)  
**Pilot Region:** FEMA Region 5 (Illinois/Midwest)

---

## About the CSU-MLP Framework

The operational CSU-MLP code is proprietary. This project reconstructs the **Hill et al. (2023) methodology** using:
- `scikit-learn` RandomForestClassifier (same architecture)
- GEFSv12 ingredients (CAPE, shear, moisture) from AWS
- EAGLE-I county-level outage data from DOE
- Spatial neighborhood mapping (40km radius per Hill et al.)
- Tree Interpreter / SHAP for explainability (per Mazurek et al. 2025)

Key references:
- Hill, Schumacher & Jirak (2023) — GEFSv12 RF severe weather [WAF]
- Alpay et al. (2020) — Outage prediction with HRRR [MDPI]
- Saki et al. (2025) — Compound events & EAGLE-I [Nature Sci Rep]
- Mazurek et al. (2025) — Tree Interpreter explainability [WAF]
- Tervo et al. (2021) — Object-based storm tracking [NHESS]

---

## Project Notebook Structure

| Notebook | Phase | Deadline |
|---|---|---|
| `01_data_ingestion.ipynb` | Download GEFSv12 + EAGLE-I | Mar 1 |
| `02_geospatial_integration.ipynb` | Spatial mapping + feature engineering | Apr 5 |
| `03_modeling.ipynb` | Random Forest training + validation | Apr 26 |
| `04_interpretability.ipynb` | Tree Interpreter + SHAP contribution maps | May 6 |

---

## Environment Setup

Run the cell below once to install all required packages.

In [1]:
# Install all project dependencies
# Run this cell once; restart kernel after

import subprocess, sys

packages = [
    # Core data science
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",

    # Weather data (GRIB2 / NetCDF)
    "xarray",
    "cfgrib",          # xarray GRIB2 engine
    "eccodes",         # GRIB2 codec (cfgrib dependency)
    "netCDF4",
    "h5py",

    # Geospatial
    "geopandas",
    "shapely",
    "pyproj",
    "fiona",
    "cartopy",

    # AWS / data access
    "boto3",
    "s3fs",            # xarray S3 backend
    "requests",

    # ML & explainability
    "shap",
    "treeinterpreter",
    "imbalanced-learn",   # SMOTE for class imbalance

    # Visualization
    "matplotlib",
    "seaborn",
    "plotly",

    # Utilities
    "tqdm",
    "joblib",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  WARNING: {pkg} failed — {result.stderr.strip()[:80]}")
    else:
        print(f"  OK")

print("\nDone. Restart the kernel before running other notebooks.")

Installing numpy...
  OK
Installing pandas...
  OK
Installing scipy...
  OK
Installing scikit-learn...
  OK
Installing xarray...
  OK
Installing cfgrib...
  OK
Installing eccodes...
  OK
Installing netCDF4...
  OK
Installing h5py...
  OK
Installing geopandas...
  OK
Installing shapely...
  OK
Installing pyproj...
  OK
Installing fiona...
  OK
Installing cartopy...
  OK
Installing boto3...
  OK
Installing s3fs...
  OK
Installing requests...
  OK
Installing shap...
  OK
Installing treeinterpreter...
  OK
Installing imbalanced-learn...
  OK
Installing matplotlib...
  OK
Installing seaborn...
  OK
Installing plotly...
  OK
Installing tqdm...
  OK
Installing joblib...
  OK

Done. Restart the kernel before running other notebooks.


## Verify Imports

In [2]:
import importlib, sys

required = {
    "numpy": "np",
    "pandas": "pd",
    "sklearn": "scikit-learn",
    "xarray": "xr",
    "geopandas": "gpd",
    "boto3": "boto3",
    "s3fs": "s3fs",
    "shap": "shap",
    "treeinterpreter": "treeinterpreter",
    "imblearn": "imbalanced-learn",
    "cartopy": "cartopy",
    "cfgrib": "cfgrib",
    "matplotlib": "matplotlib",
}

print(f"Python {sys.version}\n")
all_ok = True
for mod, label in required.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  ✓  {label:<22} {ver}")
    except ImportError:
        print(f"  ✗  {label:<22} NOT FOUND")
        all_ok = False

print("\nAll dependencies satisfied!" if all_ok else "\n⚠️ Some packages missing — re-run the install cell.")

Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]

  ✓  np                     2.4.2
  ✓  pd                     3.0.1
  ✓  scikit-learn           1.8.0
  ✓  xr                     2026.2.0
  ✓  gpd                    1.1.3
  ✓  boto3                  1.42.66
  ✓  s3fs                   2026.2.0
  ✓  shap                   0.51.0
  ✓  treeinterpreter        0.1.0
  ✓  imbalanced-learn       0.14.1
  ✓  cartopy                0.25.0
  ✓  cfgrib                 0.9.15.1
  ✓  matplotlib             3.10.8

All dependencies satisfied!


## Project Config

Central config used across all notebooks — edit paths here.

In [3]:
# ============================================================
# PROJECT CONFIGURATION — edit these paths as needed
# ============================================================
import os
from pathlib import Path

# --- Root ---
PROJECT_ROOT = Path("/data/keeling/a/tob3/Capstone2026/Testingthings")   # adjust if needed

# --- Data directories ---
DATA_RAW      = PROJECT_ROOT / "data" / "raw"
DATA_PROC     = PROJECT_ROOT / "data" / "processed"
GEFS_DIR      = DATA_RAW / "gefs"
EAGLEI_DIR    = DATA_RAW / "eaglei"
COUNTY_SHP    = DATA_RAW / "counties"     # US county shapefile

# --- Output / model ---
MODEL_DIR     = PROJECT_ROOT / "models"
OUTPUT_DIR    = PROJECT_ROOT / "outputs"

# Create dirs if they don't exist
for d in [DATA_RAW, DATA_PROC, GEFS_DIR, EAGLEI_DIR, COUNTY_SHP, MODEL_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Study domain: FEMA Region 5 state FIPS codes ---
# IL, IN, MI, MN, OH, WI
REGION5_FIPS = ["17", "18", "26", "27", "39", "55"]

# --- FEMA Region 5 bounding box (lon_min, lon_max, lat_min, lat_max) ---
DOMAIN_LON = (-97.5, -80.0)
DOMAIN_LAT = (36.5,  49.5)

# --- Training / test split ---
TRAIN_YEARS = [2021, 2022]
TEST_YEARS  = [2023]

# --- GEFSv12 config (Hill et al. 2023) ---
GEFS_RESOLUTION = 0.25   # degrees
SPATIAL_RADIUS_KM = 40   # neighborhood radius for spatial averaging
TEMPORAL_LAG_HR   = 1    # Alpay et al. 2020: 1-hr lag between wind peak & outage report

# --- Outage threshold (fraction of county customers affected = "outage event") ---
OUTAGE_THRESHOLD = 0.01  # 1% of county customers

print("Config loaded. Project root:", PROJECT_ROOT)
print("Domain:", DOMAIN_LON, DOMAIN_LAT)
print("Training years:", TRAIN_YEARS, "| Test years:", TEST_YEARS)

Config loaded. Project root: /data/keeling/a/tob3/Capstone2026/Testingthings
Domain: (-97.5, -80.0) (36.5, 49.5)
Training years: [2021, 2022] | Test years: [2023]


## Feature List (GEFSv12 Ingredients)

Based on Hill et al. (2023) + Saki et al. (2025) compound variables.

In [4]:
# ============================================================
# FEATURE DEFINITIONS
# Mirrors the "ingredients-based" approach of Hill et al. 2023
# ============================================================

# Base GEFSv12 features (GRIB2 shortNames or CFVarNames)
GEFS_FEATURES = {
    # Kinematic
    "ugrd_10m":   "10m U-wind component (m/s)",
    "vgrd_10m":   "10m V-wind component (m/s)",
    "gust_sfc":   "Surface wind gust (m/s)",
    "ugrd_shear": "0-6km bulk wind shear U-component",
    "vgrd_shear": "0-6km bulk wind shear V-component",
    # Thermodynamic
    "cape_sfc":   "Surface-based CAPE (J/kg)",
    "cin_sfc":    "Surface-based CIN (J/kg)",
    "hlcy_3km":   "0-3km Storm-Relative Helicity (m2/s2)",
    # Moisture
    "pwat_clm":   "Precipitable water (mm)",
    "apcp_sfc":   "Accumulated precipitation (mm)",
    "soilw_0_10": "0-10cm volumetric soil moisture (frac)",
}

# Derived / compound features (Saki et al. 2025)
COMPOUND_FEATURES = [
    "wind_shear_mag",   # sqrt(ugrd_shear^2 + vgrd_shear^2)
    "wind_gust_x_soilw",# gust_sfc * soilw_0_10 — wet-soil amplification
    "wind_gust_x_apcp", # gust_sfc * apcp_sfc  — rain+wind compound
    "cape_x_shear",     # CAPE * shear_mag     — supercell composite proxy
]

# Static features (land use / vegetation proxy — Tervo et al. 2021)
STATIC_FEATURES = [
    "forest_fraction",  # county-level canopy cover fraction (NLCD)
    "pop_density",      # population density (customers at risk)
]

ALL_FEATURES = list(GEFS_FEATURES.keys()) + COMPOUND_FEATURES + STATIC_FEATURES

print(f"Total features: {len(ALL_FEATURES)}")
for f in ALL_FEATURES:
    desc = GEFS_FEATURES.get(f, "(derived)")
    print(f"  {f:<25} {desc}")

Total features: 17
  ugrd_10m                  10m U-wind component (m/s)
  vgrd_10m                  10m V-wind component (m/s)
  gust_sfc                  Surface wind gust (m/s)
  ugrd_shear                0-6km bulk wind shear U-component
  vgrd_shear                0-6km bulk wind shear V-component
  cape_sfc                  Surface-based CAPE (J/kg)
  cin_sfc                   Surface-based CIN (J/kg)
  hlcy_3km                  0-3km Storm-Relative Helicity (m2/s2)
  pwat_clm                  Precipitable water (mm)
  apcp_sfc                  Accumulated precipitation (mm)
  soilw_0_10                0-10cm volumetric soil moisture (frac)
  wind_shear_mag            (derived)
  wind_gust_x_soilw         (derived)
  wind_gust_x_apcp          (derived)
  cape_x_shear              (derived)
  forest_fraction           (derived)
  pop_density               (derived)
